In [4]:
import warnings
warnings.simplefilter("ignore", UserWarning)

import os
import pandas as pd
import re

from nltk.tokenize import word_tokenize

from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel
from textwrap import dedent

In [5]:
#GCP
from google.cloud import bigquery
from google.cloud.bigquery.client import Client

## 4o-mini

# NAFO

## Getting data

In [6]:
os.environ[
    'GOOGLE_APPLICATION_CREDENTIALS'] = 'C:\\Users\\N10980695\\qutscripts\\PycharmProjects\\pilot\\DMRC_Academic_Twitter_Archive_Collector\\DATA_collector\\access_key\\dmrc-data-632c6dd69009.json'
gbq_project = 'dmrc-data'
dataset = 'ua_nafo_main'  #change the name of the dataset here
table = 'tweets'
limit = 50 #change the number of the items needed
cluster_table = 'degree5_mod_1_clusters' #change the name of modularity table here
clusters_to_exclude = ['8']

client = bigquery.Client()
bq = Client(project=gbq_project)

Things to filter out further - tweets that are too short

In [7]:
query_string = f"""select tweet_id, tweet_text
from
(select twt.tweet_id,
      twt.author_username,
      twt.tweet_text,
      twt.referenced_tweet_author_username,
      twt.referenced_tweet_author_id,
      twt.tweet_type,
      clust.modularity_class as user_class,
      ref_clust.modularity_class as reference_class
from {dataset}.{table} twt
left join {dataset}.{cluster_table} clust
on twt.author_id=clust.Id
left join {dataset}.{cluster_table} ref_clust
on twt.referenced_tweet_author_id=ref_clust.Id
where twt.reference_level = '0')
where user_class NOT IN UNNEST({clusters_to_exclude})
and
((reference_class IS NULL) or
(reference_class NOT IN UNNEST({clusters_to_exclude})))
and tweet_type != 'retweet'
and not contains_substr(tweet_text, 'MAKS_NAFO_FELLA')
LIMIT {limit}
"""

# Run query and save to dataframe
print(f"Querying '{table}' table...")
print(query_string)

Querying 'tweets' table...
select tweet_id, tweet_text
from
(select twt.tweet_id,
      twt.author_username,
      twt.tweet_text,
      twt.referenced_tweet_author_username,
      twt.referenced_tweet_author_id,
      twt.tweet_type,
      clust.modularity_class as user_class,
      ref_clust.modularity_class as reference_class
from ua_nafo_main.tweets twt
left join ua_nafo_main.degree5_mod_1_clusters clust
on twt.author_id=clust.Id
left join ua_nafo_main.degree5_mod_1_clusters ref_clust
on twt.referenced_tweet_author_id=ref_clust.Id
where twt.reference_level = '0')
where user_class NOT IN UNNEST(['8'])
and
((reference_class IS NULL) or
(reference_class NOT IN UNNEST(['8'])))
and tweet_type != 'retweet'
and not contains_substr(tweet_text, 'MAKS_NAFO_FELLA')
LIMIT 50



In [8]:
df = (bq.query(query_string).result().to_dataframe())

In [9]:
df['clean_text'] = df.tweet_text.apply(lambda x: re.sub(r'(#\w*?)\s|&', '', x))
df['clean_text'] = df.clean_text.apply(lambda x: re.sub(r'(@\w*?)\s|&', '', x))
df['clean_text'] = df.clean_text.apply(lambda x: re.sub(r'http\S+', '', x))
df['clean_tokenized'] = df.clean_text.apply(lambda x: word_tokenize(x))
df['n_tokens'] = df.clean_tokenized.apply(lambda x: len(x))
df = df.loc[df.n_tokens > 3].copy(deep=True)
df.drop(columns=['clean_tokenized', 'n_tokens', 'clean_text'], inplace=True)

## Setting up OpenAI

In [10]:
# create an openai account and get the API key from here "https://platform.openai.com/account/api-keys"
load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [11]:
MODEL='gpt-4o-mini'

In [12]:
class LabeledTweet(BaseModel):
    label: str
    text: str

class LabeledTweets(BaseModel):
    tweets: list[LabeledTweet]

In [13]:
with open('mar_2024_prompts/v6_cot/NAFO_MPE_COT_0_with_system_instruction.txt', encoding='utf-8') as f:
    cot_mpe_prompt = f.read()

In [14]:
tweets = df.tweet_text.to_list()

In [15]:
content = ''
for n,i in enumerate(tweets):
    content += f'Tweet_{n}:{i}\n'

In [16]:
def get_tweet_label(text: str):
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": dedent(cot_mpe_prompt)}, #prompt goes here
            {"role": "user", "content": text} #text to classify goes here
        ],
        response_format=LabeledTweets,
    )

    return completion.choices[0].message.parsed

In [17]:
results = get_tweet_label(content)

In [18]:
tweets = results.tweets

In [19]:
labels = [i.label for i in tweets]

In [20]:
texts = [i.text for i in tweets]

In [21]:
result_df = pd.DataFrame([texts,labels]).T

In [22]:
result_df.columns = ['tweet_text', 'label']

In [23]:
result_df

,tweet_text,label
0,One of the few cheering things this year has b...,Community work
1,#NAFO rise. Dear #NAFOfellas thanks for your s...,Community work
2,#Ukraine #NAFO #Thread\n«Electrical»\n\nUkrain...,Not applicable
3,One of the earliest known Shiba representation...,Meme creation
4,The NAFO pack of Shibas were the first inhabit...,Not applicable
5,RT @frontlinekit@nafo.uk\nThe Team-up you have...,Fundraising
6,"#MoscowOnFire ?\n\nOi, We were doing a set at ...",Play
7,Dear #NAFO #NAFOfellas\n#NAFOExpansionIsNonNeg...,Mobilising
8,#Sunday #Bloody Sunday (Live From #RedRocks Am...,News and content curation
9,@fellarequests @Kama_Kamilia @Official_NAFO \n...,Fundraising


## Error to fix - when merging on texts, 6 out of 30 tweets lost

In [24]:
twt_str = ''.join(result_df.tweet_text.to_list())

In [25]:
lbl_str = ''.join(result_df.label.to_list())

In [26]:
labeled_df = df.merge(result_df, on='tweet_text')

In [27]:
labeled_df

,tweet_id,tweet_text,label
0,1609294347316387842,One of the few cheering things this year has b...,Community work
1,1609225888121475072,#NAFO rise. Dear #NAFOfellas thanks for your s...,Community work
2,1609247119222931457,#Ukraine #NAFO #Thread\n«Electrical»\n\nUkrain...,Not applicable
3,1609289597124579329,One of the earliest known Shiba representation...,Meme creation
4,1609276346424852481,The NAFO pack of Shibas were the first inhabit...,Not applicable
5,1609302402061975555,RT @frontlinekit@nafo.uk\nThe Team-up you have...,Fundraising
6,1609344870958927874,"#MoscowOnFire ?\n\nOi, We were doing a set at ...",Play
7,1609236919715594240,Dear #NAFO #NAFOfellas\n#NAFOExpansionIsNonNeg...,Mobilising
8,1609390420840288257,#Sunday #Bloody Sunday (Live From #RedRocks Am...,News and content curation
9,1609235941931028486,@fellarequests @Kama_Kamilia @Official_NAFO \n...,Fundraising


## Assuming the error is fixed, let's try and calculate the costs

In [28]:
import tiktoken

In [29]:
MODEL = 'gpt-4o-mini'
PRICE_IN_PT = 0.150 / 1000000
PRICE_OUT_PT = 0.600 / 1000000
N_NAFO = 1315982
N_ESC = 125569 + 38504

In [30]:
#MODEL = 'gpt-4o'
#PRICE_IN_PT = 2.5 / 1000000
#PRICE_OUT_PT = 10 / 1000000

In [31]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [32]:
with open(f'mar_2024_prompts/v6_cot/NAFO_MPE_COT_0_with_system_instruction.txt', encoding='utf-8') as f:
    cot = f.read()
n_tweets = result_df.shape[0]
tweet_input = content
total_in_tokens = num_tokens_from_string(tweet_input + cot, MODEL)
thirty_in_price = total_in_tokens * PRICE_IN_PT
avg_one_tweet_input = thirty_in_price / n_tweets
avg_one_tweet_input

2.46078947368421e-05

In [33]:
total_out_tokens = num_tokens_from_string(twt_str + lbl_str, MODEL)
thirty_out_price = total_out_tokens * PRICE_OUT_PT
avg_one_tweet_output = thirty_out_price / n_tweets
avg_one_tweet_output

4.312105263157894e-05

In [34]:
total_one = avg_one_tweet_output + avg_one_tweet_input

In [35]:
total_one * N_NAFO

89.13007561578947

# ESC

In [36]:
os.environ[
    'GOOGLE_APPLICATION_CREDENTIALS'] = 'C:\\Users\\N10980695\\qutscripts\\PycharmProjects\\pilot\\DMRC_Academic_Twitter_Archive_Collector\\DATA_collector\\access_key\\dmrc-data-632c6dd69009.json'
gbq_project = 'dmrc-data'
dataset = 'ua_eurovision_2023'  #change the name of the dataset here
table = 'tweets'
limit = 50  #change the number of the items needed
cluster_table = 'degree2_mod_05_clusters'  #change the name of modularity table here
clusters_to_exclude = ['196', '191']

client = bigquery.Client()
bq = Client(project=gbq_project)
query_string = f"""select tweet_id, tweet_text
from
(select twt.tweet_id,
      twt.author_username,
      twt.tweet_text,
      twt.referenced_tweet_author_username,
      twt.referenced_tweet_author_id,
      twt.tweet_type,
      clust.modularity_class as user_class,
      ref_clust.modularity_class as reference_class
from {dataset}.{table} twt
left join {dataset}.{cluster_table} clust
on twt.author_id=clust.Id
left join {dataset}.{cluster_table} ref_clust
on twt.referenced_tweet_author_id=ref_clust.Id
where twt.reference_level = '0')
where user_class NOT IN UNNEST({clusters_to_exclude})
and
((reference_class IS NULL) or
(reference_class NOT IN UNNEST({clusters_to_exclude})))
and tweet_type != 'retweet'
and not contains_substr(tweet_text, 'MAKS_NAFO_FELLA')
LIMIT {limit}
"""

# Run query and save to dataframe
print(f"Querying '{table}' table...")
print(query_string)

Querying 'tweets' table...
select tweet_id, tweet_text
from
(select twt.tweet_id,
      twt.author_username,
      twt.tweet_text,
      twt.referenced_tweet_author_username,
      twt.referenced_tweet_author_id,
      twt.tweet_type,
      clust.modularity_class as user_class,
      ref_clust.modularity_class as reference_class
from ua_eurovision_2023.tweets twt
left join ua_eurovision_2023.degree2_mod_05_clusters clust
on twt.author_id=clust.Id
left join ua_eurovision_2023.degree2_mod_05_clusters ref_clust
on twt.referenced_tweet_author_id=ref_clust.Id
where twt.reference_level = '0')
where user_class NOT IN UNNEST(['196', '191'])
and
((reference_class IS NULL) or
(reference_class NOT IN UNNEST(['196', '191'])))
and tweet_type != 'retweet'
and not contains_substr(tweet_text, 'MAKS_NAFO_FELLA')
LIMIT 50



In [37]:
df = (bq.query(query_string).result().to_dataframe())
df['clean_text'] = df.tweet_text.apply(lambda x: re.sub(r'(#\w*?)\s|&', '', x))
df['clean_text'] = df.clean_text.apply(lambda x: re.sub(r'(@\w*?)\s|&', '', x))
df['clean_text'] = df.clean_text.apply(lambda x: re.sub(r'http\S+', '', x))
df['clean_tokenized'] = df.clean_text.apply(lambda x: word_tokenize(x))
df['n_tokens'] = df.clean_tokenized.apply(lambda x: len(x))
df = df.loc[df.n_tokens > 3].copy(deep=True)
df.drop(columns=['clean_tokenized', 'n_tokens', 'clean_text'], inplace=True)

In [38]:
MODEL = 'gpt-4o-mini'
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

class LabeledTweet(BaseModel):
    label: str
    text: str


class LabeledTweets(BaseModel):
    tweets: list[LabeledTweet]


with open('mar_2024_prompts/v6_cot/ESC_MPE_COT_0_with_system_instruction.txt', encoding='utf-8') as f:
    cot_mpe_prompt = f.read()
tweets = df.tweet_text.to_list()
content = ''
for n, i in enumerate(tweets):
    content += f'Tweet_{n}:{i}\n'


def get_tweet_label(text: str):
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": dedent(cot_mpe_prompt)},  #prompt goes here
            {"role": "user", "content": text}  #text to classify goes here
        ],
        response_format=LabeledTweets,
    )

    return completion.choices[0].message.parsed


results = get_tweet_label(content)
tweets = results.tweets
labels = [i.label for i in tweets]
texts = [i.text for i in tweets]
result_df = pd.DataFrame([texts, labels]).T
result_df.columns = ['tweet_text', 'label']
twt_str = ''.join(result_df.tweet_text.to_list())
lbl_str = ''.join(result_df.label.to_list())
labeled_df = df.merge(result_df, on='tweet_text')
labeled_df

,tweet_id,tweet_text,label
0,1666195110738948099,My top 3 each Eurovision since 2008\n\n08 🇦🇲🇺🇦...,Knowledge performance
1,1666189575264378882,My top 3 each Eurovision since 2000\n\n00 🇩🇰🇸🇪...,Knowledge performance
2,1666087929146638337,My top 3 each Eurovision since 2000\n\n00🇸🇪🇭🇷🇷...,Knowledge performance
3,1666559703390797829,My top 3 each Eurovision since 2000\n\n00 🇷🇺🇱🇻...,Knowledge performance
4,1666199515760582684,my top 3 each eurovision since 2000\n\n00 xxx\...,Knowledge performance
5,1666740978823335937,The only one I care about is Duncan Laurence a...,Expressing emotions
6,1666175508122415104,I have not looked as far back as 2000 so - \nm...,Knowledge performance
7,1666245745253687298,Manifesting my winners of the next Eurovision ...,Knowledge performance
8,1666166028877942787,Manifesting my winners of the next Eurovision ...,Knowledge performance
9,1666302123179495424,Top 3 in eurovision since 2010 since I haven't...,Knowledge performance


In [40]:
MODEL = 'gpt-4o-mini'
PRICE_IN_PT = 0.150 / 1000000
PRICE_OUT_PT = 0.600 / 1000000
N_NAFO = 1315982
N_ESC = 125569 + 38504


#MODEL = 'gpt-4o'
#PRICE_IN_PT = 2.5 / 1000000
#PRICE_OUT_PT = 10 / 1000000
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens


with open(f'mar_2024_prompts/v6_cot/ESC_MPE_COT_0_with_system_instruction.txt', encoding='utf-8') as f:
    cot = f.read()
n_tweets = result_df.shape[0]
tweet_input = content
total_in_tokens = num_tokens_from_string(tweet_input + cot, MODEL)
thirty_in_price = total_in_tokens * PRICE_IN_PT
avg_one_tweet_input = thirty_in_price / n_tweets
total_out_tokens = num_tokens_from_string(twt_str + lbl_str, MODEL)
thirty_out_price = total_out_tokens * PRICE_OUT_PT
avg_one_tweet_output = thirty_out_price / n_tweets
total_one = avg_one_tweet_output + avg_one_tweet_input
total_one * N_ESC

22.507397412499998

# 4o

In [41]:
# NAFO
## Getting data
os.environ[
    'GOOGLE_APPLICATION_CREDENTIALS'] = 'C:\\Users\\N10980695\\qutscripts\\PycharmProjects\\pilot\\DMRC_Academic_Twitter_Archive_Collector\\DATA_collector\\access_key\\dmrc-data-632c6dd69009.json'
gbq_project = 'dmrc-data'
dataset = 'ua_nafo_main'  #change the name of the dataset here
table = 'tweets'
limit = 50  #change the number of the items needed
cluster_table = 'degree5_mod_1_clusters'  #change the name of modularity table here
clusters_to_exclude = ['8']

client = bigquery.Client()
bq = Client(project=gbq_project)
query_string = f"""select tweet_id, tweet_text
from
(select twt.tweet_id,
      twt.author_username,
      twt.tweet_text,
      twt.referenced_tweet_author_username,
      twt.referenced_tweet_author_id,
      twt.tweet_type,
      clust.modularity_class as user_class,
      ref_clust.modularity_class as reference_class
from {dataset}.{table} twt
left join {dataset}.{cluster_table} clust
on twt.author_id=clust.Id
left join {dataset}.{cluster_table} ref_clust
on twt.referenced_tweet_author_id=ref_clust.Id
where twt.reference_level = '0')
where user_class NOT IN UNNEST({clusters_to_exclude})
and
((reference_class IS NULL) or
(reference_class NOT IN UNNEST({clusters_to_exclude})))
and tweet_type != 'retweet'
and not contains_substr(tweet_text, 'MAKS_NAFO_FELLA')
LIMIT {limit}
"""

# Run query and save to dataframe
print(f"Querying '{table}' table...")
print(query_string)
df = (bq.query(query_string).result().to_dataframe())
df['clean_text'] = df.tweet_text.apply(lambda x: re.sub(r'(#\w*?)\s|&', '', x))
df['clean_text'] = df.clean_text.apply(lambda x: re.sub(r'(@\w*?)\s|&', '', x))
df['clean_text'] = df.clean_text.apply(lambda x: re.sub(r'http\S+', '', x))
df['clean_tokenized'] = df.clean_text.apply(lambda x: word_tokenize(x))
df['n_tokens'] = df.clean_tokenized.apply(lambda x: len(x))
df = df.loc[df.n_tokens > 3].copy(deep=True)
df.drop(columns=['clean_tokenized', 'n_tokens', 'clean_text'], inplace=True)
## Setting up OpenAI
# create an openai account and get the API key from here "https://platform.openai.com/account/api-keys"
load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = 'gpt-4o'


class LabeledTweet(BaseModel):
    label: str
    text: str


class LabeledTweets(BaseModel):
    tweets: list[LabeledTweet]


with open('mar_2024_prompts/v6_cot/NAFO_MPE_COT_0_with_system_instruction.txt', encoding='utf-8') as f:
    cot_mpe_prompt = f.read()
tweets = df.tweet_text.to_list()
content = ''
for n, i in enumerate(tweets):
    content += f'Tweet_{n}:{i}\n'


def get_tweet_label(text: str):
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": dedent(cot_mpe_prompt)},  #prompt goes here
            {"role": "user", "content": text}  #text to classify goes here
        ],
        response_format=LabeledTweets,
    )

    return completion.choices[0].message.parsed


results = get_tweet_label(content)
tweets = results.tweets
labels = [i.label for i in tweets]
texts = [i.text for i in tweets]
result_df = pd.DataFrame([texts, labels]).T
result_df.columns = ['tweet_text', 'label']
result_df

Querying 'tweets' table...
select tweet_id, tweet_text
from
(select twt.tweet_id,
      twt.author_username,
      twt.tweet_text,
      twt.referenced_tweet_author_username,
      twt.referenced_tweet_author_id,
      twt.tweet_type,
      clust.modularity_class as user_class,
      ref_clust.modularity_class as reference_class
from ua_nafo_main.tweets twt
left join ua_nafo_main.degree5_mod_1_clusters clust
on twt.author_id=clust.Id
left join ua_nafo_main.degree5_mod_1_clusters ref_clust
on twt.referenced_tweet_author_id=ref_clust.Id
where twt.reference_level = '0')
where user_class NOT IN UNNEST(['8'])
and
((reference_class IS NULL) or
(reference_class NOT IN UNNEST(['8'])))
and tweet_type != 'retweet'
and not contains_substr(tweet_text, 'MAKS_NAFO_FELLA')
LIMIT 50



,tweet_text,label
0,Tweet_0:One of the few cheering things this ye...,Shitposting
1,Tweet_1:#NAFO rise. Dear #NAFOfellas thanks fo...,Community work
2,Tweet_2:#Ukraine #NAFO #Thread\n«Electrical»\n...,Audiencing
3,Tweet_3:One of the earliest known Shiba repres...,Play
4,Tweet_4:The NAFO pack of Shibas were the first...,Play
5,Tweet_5:RT @frontlinekit@nafo.uk\nThe Team-up ...,Fundraising
6,"Tweet_6:#MoscowOnFire ?\n\nOi, We were doing a...",Play
7,Tweet_7:Dear #NAFO #NAFOfellas\n#NAFOExpansion...,Mobilising
8,Tweet_8:#Sunday #Bloody Sunday (Live From #Red...,Fundraising
9,Tweet_9:@fellarequests @Kama_Kamilia @Official...,Membership requests


In [42]:

result_df['tweet_text'] = result_df.tweet_text.str.replace(r'Tweet_\d+:', '', regex=True)

In [43]:
twt_str = ''.join(result_df.tweet_text.to_list())
lbl_str = ''.join(result_df.label.to_list())
labeled_df = df.merge(result_df, on='tweet_text')

In [44]:
labeled_df

,tweet_id,tweet_text,label
0,1609294347316387842,One of the few cheering things this year has b...,Shitposting
1,1609225888121475072,#NAFO rise. Dear #NAFOfellas thanks for your s...,Community work
2,1609247119222931457,#Ukraine #NAFO #Thread\n«Electrical»\n\nUkrain...,Audiencing
3,1609289597124579329,One of the earliest known Shiba representation...,Play
4,1609276346424852481,The NAFO pack of Shibas were the first inhabit...,Play
5,1609302402061975555,RT @frontlinekit@nafo.uk\nThe Team-up you have...,Fundraising
6,1609344870958927874,"#MoscowOnFire ?\n\nOi, We were doing a set at ...",Play
7,1609236919715594240,Dear #NAFO #NAFOfellas\n#NAFOExpansionIsNonNeg...,Mobilising
8,1609390420840288257,#Sunday #Bloody Sunday (Live From #RedRocks Am...,Fundraising
9,1609235941931028486,@fellarequests @Kama_Kamilia @Official_NAFO \n...,Membership requests


In [45]:
N_NAFO = 1315982
N_ESC = 125569 + 38504


MODEL = 'gpt-4o'
PRICE_IN_PT = 2.5 / 1000000
PRICE_OUT_PT = 10 / 1000000

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens


with open(f'mar_2024_prompts/v6_cot/NAFO_MPE_COT_0_with_system_instruction.txt', encoding='utf-8') as f:
    cot = f.read()
n_tweets = result_df.shape[0]
tweet_input = content
total_in_tokens = num_tokens_from_string(tweet_input + cot, MODEL)
thirty_in_price = total_in_tokens * PRICE_IN_PT
avg_one_tweet_input = thirty_in_price / n_tweets

total_out_tokens = num_tokens_from_string(twt_str + lbl_str, MODEL)
thirty_out_price = total_out_tokens * PRICE_OUT_PT
avg_one_tweet_output = thirty_out_price / n_tweets

total_one = avg_one_tweet_output + avg_one_tweet_input
total_one * N_NAFO

1490.6959260526316

# ESC

In [46]:
# ESC
os.environ[
    'GOOGLE_APPLICATION_CREDENTIALS'] = 'C:\\Users\\N10980695\\qutscripts\\PycharmProjects\\pilot\\DMRC_Academic_Twitter_Archive_Collector\\DATA_collector\\access_key\\dmrc-data-632c6dd69009.json'
gbq_project = 'dmrc-data'
dataset = 'ua_eurovision_2023'  #change the name of the dataset here
table = 'tweets'
limit = 50  #change the number of the items needed
cluster_table = 'degree2_mod_05_clusters'  #change the name of modularity table here
clusters_to_exclude = ['196', '191']

client = bigquery.Client()
bq = Client(project=gbq_project)
query_string = f"""select tweet_id, tweet_text
from
(select twt.tweet_id,
      twt.author_username,
      twt.tweet_text,
      twt.referenced_tweet_author_username,
      twt.referenced_tweet_author_id,
      twt.tweet_type,
      clust.modularity_class as user_class,
      ref_clust.modularity_class as reference_class
from {dataset}.{table} twt
left join {dataset}.{cluster_table} clust
on twt.author_id=clust.Id
left join {dataset}.{cluster_table} ref_clust
on twt.referenced_tweet_author_id=ref_clust.Id
where twt.reference_level = '0')
where user_class NOT IN UNNEST({clusters_to_exclude})
and
((reference_class IS NULL) or
(reference_class NOT IN UNNEST({clusters_to_exclude})))
and tweet_type != 'retweet'
and not contains_substr(tweet_text, 'MAKS_NAFO_FELLA')
LIMIT {limit}
"""

# Run query and save to dataframe
print(f"Querying '{table}' table...")
print(query_string)
df = (bq.query(query_string).result().to_dataframe())
df['clean_text'] = df.tweet_text.apply(lambda x: re.sub(r'(#\w*?)\s|&', '', x))
df['clean_text'] = df.clean_text.apply(lambda x: re.sub(r'(@\w*?)\s|&', '', x))
df['clean_text'] = df.clean_text.apply(lambda x: re.sub(r'http\S+', '', x))
df['clean_tokenized'] = df.clean_text.apply(lambda x: word_tokenize(x))
df['n_tokens'] = df.clean_tokenized.apply(lambda x: len(x))
df = df.loc[df.n_tokens > 3].copy(deep=True)
df.drop(columns=['clean_tokenized', 'n_tokens', 'clean_text'], inplace=True)
MODEL = 'gpt-4o'
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


class LabeledTweet(BaseModel):
    label: str
    text: str


class LabeledTweets(BaseModel):
    tweets: list[LabeledTweet]


with open('mar_2024_prompts/v6_cot/ESC_MPE_COT_0_with_system_instruction.txt', encoding='utf-8') as f:
    cot_mpe_prompt = f.read()
tweets = df.tweet_text.to_list()
content = ''
for n, i in enumerate(tweets):
    content += f'Tweet_{n}:{i}\n'


def get_tweet_label(text: str):
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": dedent(cot_mpe_prompt)},  #prompt goes here
            {"role": "user", "content": text}  #text to classify goes here
        ],
        response_format=LabeledTweets,
    )

    return completion.choices[0].message.parsed


results = get_tweet_label(content)
tweets = results.tweets
labels = [i.label for i in tweets]
texts = [i.text for i in tweets]
result_df = pd.DataFrame([texts, labels]).T
result_df.columns = ['tweet_text', 'label']
result_df['tweet_text'] = result_df.tweet_text.str.replace(r'Tweet_\d+:', '', regex=True)
twt_str = ''.join(result_df.tweet_text.to_list())
lbl_str = ''.join(result_df.label.to_list())
labeled_df = df.merge(result_df, on='tweet_text')
labeled_df

Querying 'tweets' table...
select tweet_id, tweet_text
from
(select twt.tweet_id,
      twt.author_username,
      twt.tweet_text,
      twt.referenced_tweet_author_username,
      twt.referenced_tweet_author_id,
      twt.tweet_type,
      clust.modularity_class as user_class,
      ref_clust.modularity_class as reference_class
from ua_eurovision_2023.tweets twt
left join ua_eurovision_2023.degree2_mod_05_clusters clust
on twt.author_id=clust.Id
left join ua_eurovision_2023.degree2_mod_05_clusters ref_clust
on twt.referenced_tweet_author_id=ref_clust.Id
where twt.reference_level = '0')
where user_class NOT IN UNNEST(['196', '191'])
and
((reference_class IS NULL) or
(reference_class NOT IN UNNEST(['196', '191'])))
and tweet_type != 'retweet'
and not contains_substr(tweet_text, 'MAKS_NAFO_FELLA')
LIMIT 50



,tweet_id,tweet_text,label
0,1666195110738948099,My top 3 each Eurovision since 2008\n\n08 🇦🇲🇺🇦...,Audiencing
1,1666189575264378882,My top 3 each Eurovision since 2000\n\n00 🇩🇰🇸🇪...,Audiencing
2,1666087929146638337,My top 3 each Eurovision since 2000\n\n00🇸🇪🇭🇷🇷...,Audiencing
3,1666559703390797829,My top 3 each Eurovision since 2000\n\n00 🇷🇺🇱🇻...,Audiencing
4,1666199515760582684,my top 3 each eurovision since 2000\n\n00 xxx\...,Audiencing
...,...,...,...
78,1666836002692911104,"#Eurovision personal 3rd place-table, 2023 upd...",Audiencing
79,1666835313543585795,"#Eurovision personal runners-up-table, 2023 up...",Audiencing
80,1666901805396025357,"Join JAMALA, a Ukrainian singer/songwriter and...",News and content curation
81,1666863850472210438,remember the eurofan who had a patreon for exc...,Audiencing


In [47]:

N_NAFO = 1315982
N_ESC = 125569 + 38504


MODEL = 'gpt-4o'
PRICE_IN_PT = 2.5 / 1000000
PRICE_OUT_PT = 10 / 1000000

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens


with open(f'mar_2024_prompts/v6_cot/ESC_MPE_COT_0_with_system_instruction.txt', encoding='utf-8') as f:
    cot = f.read()
n_tweets = result_df.shape[0]
tweet_input = content
total_in_tokens = num_tokens_from_string(tweet_input + cot, MODEL)
thirty_in_price = total_in_tokens * PRICE_IN_PT
avg_one_tweet_input = thirty_in_price / n_tweets
total_out_tokens = num_tokens_from_string(twt_str + lbl_str, MODEL)
thirty_out_price = total_out_tokens * PRICE_OUT_PT
avg_one_tweet_output = thirty_out_price / n_tweets
total_one = avg_one_tweet_output + avg_one_tweet_input
total_one * N_ESC

326.0427237765958